<a href="https://colab.research.google.com/github/shreyaghora/Bank_Marketing_Project/blob/main/Code/Smote_Friedman_Test_Among_12_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install lightgbm
!pip install catboost
!pip install xgboost
!pip install specificity
!pip install imbalanced-learn

!pip install ipython-autotime
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 36.9 s (started: 2026-05-09 15:54:22 +00:00)


In [ ]:
import numpy as np
import pandas as pd
import time
from scipy.stats import uniform


from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate

from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix, balanced_accuracy_score

time: 3.31 ms (started: 2026-05-09 15:54:59 +00:00)


In [ ]:
df = pd.read_csv("bank-additional-full.csv", sep = ";")
df

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,...,2,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes


time: 250 ms (started: 2026-05-09 15:54:59 +00:00)


In [ ]:
# 3. Drop Leakage Feature
df.drop('duration', axis=1, inplace=True)

# 4. Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])


# 5. Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# 6. One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)
X_shuffled = df.drop('y', axis=1)
y_shuffled = df['y']

# 7. Prepare your data
X_raw = df.drop('y', axis=1)
y_raw = df['y']

# 9. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# 8. Apply SMOTE
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)


time: 651 ms (started: 2026-05-09 15:54:59 +00:00)


In [ ]:
def hyper_parameter_tune(model, param_dist, X_train, y_train):

    search = RandomizedSearchCV(
          model,
          param_distributions=param_dist,
          n_iter=5,
          cv=5,
          refit='roc_auc',
          n_jobs=-1,
          random_state=42,
          return_train_score=False
          )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    print("Best parameters:", search.best_params_)
    print("Best CV accuracy: {:.4f}".format(search.best_score_))
    print("Test accuracy: {:.4f}".format(best_model.score(X_test, y_test)))

    return best_model

time: 1.46 ms (started: 2026-05-09 15:55:00 +00:00)


In [ ]:
def evaluate_models(name, model, X_shuffled, y_shuffled):

    result = []

    print(f"\nTrain Model: {name}\n")

    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
      X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
      y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

      # Train
      start_time = time.time()
      model.fit(X_train, y_train)
      training_time = time.time() - start_time

      # Predict
      y_pred = model.predict(X_test)

      # Metrics
      acc = accuracy_score(y_test, y_pred)
      prec = precision_score(y_test, y_pred, zero_division=0)
      rec = recall_score(y_test, y_pred, zero_division=0)
      f1 = f1_score(y_test, y_pred, zero_division=0)
      cm = confusion_matrix(y_test, y_pred)
      tn, fp, fn, tp = cm.ravel()
      specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
      fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
      gm = np.sqrt(rec * specificity)
      auc = roc_auc_score(y_test, y_pred)
      mcc = matthews_corrcoef(y_test, y_pred)
      kappa = cohen_kappa_score(y_test, y_pred)
      balanced_acc = balanced_accuracy_score(y_test, y_pred)

      # Store fold results
      result.append({
          "Fold": fold+1,
          "Classifier": name,
          "Accuracy": acc,
          "Precision": prec,
          "Recall": rec,
          "Specificity": specificity,
          "F1": f1,
          "GM": gm,
          "FPR": fpr,
          "AUC": auc,
          "MCC": mcc,
          "Kappa": kappa,
          "Balanced Accuracy": balanced_acc,
          "Training Time (s)": training_time
      })
    return pd.DataFrame(result)

time: 3.97 ms (started: 2026-05-09 15:55:00 +00:00)


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

name = "Gradient Boosting (GBM)" # change as per model

# Model
model = GradientBoostingClassifier() # change as per model

# Parameters, change as per model
param_dist = {
        'n_estimators': [21, 51, 81],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [1, 2, 3, 5],
        'subsample': [0.6, 0.8, 0.9]
    }

best_model = hyper_parameter_tune(model, param_dist, X_train_sm, y_train_sm)

# run
df1 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df1.to_csv(f"{name} result.csv")
df1

Best parameters: {'subsample': 0.9, 'n_estimators': 81, 'max_depth': 3, 'learning_rate': 0.05}
Best CV accuracy: 0.8297
Test accuracy: 0.8572

Train Model: Gradient Boosting (GBM)



,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,Gradient Boosting (GBM),0.821166,0.874214,0.753898,0.889617,0.809610,0.818951,0.110383,0.821758,0.649008,0.642737,0.821758,9.825819
1,2,Gradient Boosting (GBM),0.834331,0.885535,0.769414,0.899760,0.823401,0.832038,0.100240,0.834587,0.674694,0.668828,0.834587,8.765660
2,3,Gradient Boosting (GBM),0.846299,0.888889,0.786976,0.904119,0.834834,0.843516,0.095881,0.845547,0.696590,0.692100,0.845547,11.039561
3,4,Gradient Boosting (GBM),0.829544,0.880573,0.760399,0.897959,0.816086,0.826321,0.102041,0.829179,0.664997,0.658832,0.829179,11.032588
4,5,Gradient Boosting (GBM),0.830740,0.885859,0.765399,0.898193,0.821235,0.829142,0.101807,0.831796,0.668522,0.662137,0.831796,9.552463
5,6,Gradient Boosting (GBM),0.830569,0.890244,0.760672,0.902923,0.820373,0.828751,0.097077,0.831798,0.669245,0.661902,0.831798,9.747881
6,7,Gradient Boosting (GBM),0.827834,0.877152,0.763884,0.892244,0.816609,0.825573,0.107756,0.828064,0.661396,0.655823,0.828064,8.907791
7,8,Gradient Boosting (GBM),0.842024,0.881920,0.786331,0.896680,0.831387,0.839695,0.103320,0.841505,0.687685,0.683701,0.841505,9.329226
8,9,Gradient Boosting (GBM),0.825782,0.874461,0.761270,0.890448,0.813949,0.823330,0.109552,0.825859,0.657158,0.651618,0.825859,9.525303
9,10,Gradient Boosting (GBM),0.842708,0.882655,0.781939,0.900735,0.829250,0.839238,0.099265,0.841337,0.688756,0.684436,0.841337,9.318710


time: 3min 15s (started: 2026-05-09 15:55:00 +00:00)


In [ ]:
from sklearn.linear_model import LogisticRegression

name ="CatBoost"

# Model
model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_strength=1,
    bagging_temperature=1,
    verbose=100
)

param_dist = {
    'l2_leaf_reg': [1, 3, 5, 7, 9],
    'bagging_temperature': [0, 1, 5],
    'random_strength': [1, 2, 5]
}

best_model = hyper_parameter_tune(model, param_dist, X_train_sm, y_train_sm)

# run
df5 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df5.to_csv(f"{name} result.csv")
df5

0:	learn: 0.7180373	total: 69.6ms	remaining: 13.8s
100:	learn: 0.8868354	total: 2.06s	remaining: 2.02s
199:	learn: 0.9111130	total: 3.94s	remaining: 0us
Best parameters: {'random_strength': 5, 'l2_leaf_reg': 3, 'bagging_temperature': 5}
Best CV accuracy: 0.8969
Test accuracy: 0.8806

Train Model: CatBoost

0:	learn: 0.7396326	total: 22ms	remaining: 4.38s
100:	learn: 0.8876541	total: 2.87s	remaining: 2.81s
199:	learn: 0.9122547	total: 4.79s	remaining: 0us
0:	learn: 0.7397846	total: 19.9ms	remaining: 3.95s
100:	learn: 0.8852985	total: 1.76s	remaining: 1.73s
199:	learn: 0.9109249	total: 3.49s	remaining: 0us
0:	learn: 0.7383788	total: 18.9ms	remaining: 3.75s
100:	learn: 0.8876351	total: 1.8s	remaining: 1.77s
199:	learn: 0.9091773	total: 3.66s	remaining: 0us
0:	learn: 0.7405064	total: 18.4ms	remaining: 3.67s
100:	learn: 0.8880150	total: 2.84s	remaining: 2.78s
199:	learn: 0.9123687	total: 5.08s	remaining: 0us
0:	learn: 0.7392907	total: 19.2ms	remaining: 3.83s
100:	learn: 0.8863243	total: 1.8

,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,CatBoost,0.899641,0.923022,0.873898,0.925836,0.897789,0.899493,0.074164,0.899867,0.800514,0.799361,0.899867,4.974266
1,2,CatBoost,0.905112,0.931810,0.875000,0.935462,0.902512,0.904726,0.064538,0.905231,0.811788,0.810267,0.905231,3.677731
2,3,CatBoost,0.913319,0.940415,0.880152,0.945645,0.909286,0.912311,0.054355,0.912899,0.828130,0.826468,0.912899,3.864581
3,4,CatBoost,0.907847,0.934384,0.876246,0.939116,0.904382,0.907136,0.060884,0.907681,0.817197,0.815629,0.907681,5.299388
4,5,CatBoost,0.907847,0.936155,0.878492,0.938151,0.906407,0.907832,0.061849,0.908322,0.817431,0.815832,0.908322,3.805868
5,6,CatBoost,0.903573,0.936934,0.868908,0.939457,0.901639,0.903494,0.060543,0.904182,0.809542,0.807335,0.904182,3.788550
6,7,CatBoost,0.903573,0.928752,0.874957,0.932395,0.901053,0.903220,0.067605,0.903676,0.808551,0.807184,0.903676,5.025595
7,8,CatBoost,0.913831,0.934617,0.888160,0.939024,0.910796,0.913238,0.060976,0.913592,0.828574,0.827568,0.913592,3.668039
8,9,CatBoost,0.898102,0.928047,0.863388,0.932900,0.894551,0.897471,0.067100,0.898144,0.798165,0.796221,0.898144,3.759807
9,10,CatBoost,0.907506,0.932412,0.873994,0.939505,0.902258,0.906158,0.060495,0.906750,0.816187,0.814651,0.906750,4.994874


time: 2min 18s (started: 2026-05-09 15:58:16 +00:00)


In [ ]:
from sklearn.linear_model import LogisticRegression

name ="Multi Layer Perceptron"

# Model
model = MLPClassifier(max_iter=100, random_state=42)

param_dist = {
    'hidden_layer_sizes': [(100,), (100,50), (150,100)],
    'activation': ['relu'],
    'solver': ['adam'],
    'learning_rate_init': [0.0001, 0.001],
    'alpha': [0.0001, 0.001],
}

best_model = hyper_parameter_tune(model, param_dist, X_train_sm, y_train_sm)

# run
df2 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df2.to_csv(f"{name} result.csv")
df2

Best parameters: {'solver': 'adam', 'learning_rate_init': 0.0001, 'hidden_layer_sizes': (100,), 'alpha': 0.0001, 'activation': 'relu'}
Best CV accuracy: 0.8519
Test accuracy: 0.8491

Train Model: Multi Layer Perceptron



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,Multi Layer Perceptron,0.853137,0.894082,0.804068,0.903070,0.846689,0.852132,0.096930,0.853569,0.710178,0.706512,0.853569,32.680847
1,2,Multi Layer Perceptron,0.864592,0.899702,0.821866,0.907655,0.859025,0.863696,0.092345,0.864761,0.732037,0.729273,0.864761,40.812016
2,3,Multi Layer Perceptron,0.822705,0.770310,0.913059,0.734639,0.835632,0.819004,0.265361,0.823849,0.657224,0.646181,0.823849,32.113349
3,4,Multi Layer Perceptron,0.816379,0.954883,0.662083,0.969048,0.781973,0.800993,0.030952,0.815565,0.663895,0.632153,0.815565,36.761242
4,5,Multi Layer Perceptron,0.850573,0.936693,0.756984,0.947186,0.837305,0.846761,0.052814,0.852085,0.715640,0.701994,0.852085,31.676980
5,6,Multi Layer Perceptron,0.839118,0.938740,0.731429,0.950592,0.822218,0.833840,0.049408,0.841010,0.697073,0.679384,0.841010,29.040749
6,7,Multi Layer Perceptron,0.785433,0.723048,0.927768,0.642073,0.812715,0.771813,0.357927,0.784921,0.595001,0.570423,0.784921,40.203327
7,8,Multi Layer Perceptron,0.817405,0.758552,0.926130,0.710705,0.834007,0.811298,0.289295,0.818418,0.651250,0.635527,0.818418,26.464102
8,9,Multi Layer Perceptron,0.855873,0.898357,0.802937,0.908935,0.847971,0.854294,0.091065,0.855936,0.715840,0.711782,0.855936,40.676577
9,10,Multi Layer Perceptron,0.818260,0.962371,0.653483,0.975602,0.778403,0.798460,0.024398,0.814542,0.667899,0.633673,0.814542,40.099801


time: 22min 40s (started: 2026-05-09 16:00:34 +00:00)


In [ ]:
from sklearn.svm import LinearSVC

name = "Support Vector Machine" # change as per model

# Model
model = LinearSVC(random_state=42) # change as per model

# Parameters, change as per model
param_dist = {
        'C': [10, 50, 70]
    }

best_model = hyper_parameter_tune(model, param_dist, X_train_sm, y_train_sm)

# run
df3 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df3.to_csv(f"{name} result.csv")
df3

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=5. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters: {'C': 10}
Best CV accuracy: 0.8549
Test accuracy: 0.8471

Train Model: Support Vector Machine



,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,Support Vector Machine,0.854847,0.885222,0.818305,0.892032,0.850449,0.854374,0.107968,0.855168,0.711935,0.709865,0.855168,1.196126
1,2,Support Vector Machine,0.861344,0.884546,0.832425,0.890491,0.857694,0.860969,0.109509,0.861458,0.724015,0.722748,0.861458,1.006872
2,3,Support Vector Machine,0.870747,0.897723,0.833045,0.907495,0.864175,0.869473,0.092505,0.870270,0.743102,0.741216,0.870270,0.807490
3,4,Support Vector Machine,0.855531,0.882506,0.818494,0.892177,0.849296,0.854542,0.107823,0.855336,0.712802,0.710943,0.855336,0.769480
4,5,Support Vector Machine,0.860147,0.891028,0.825648,0.895761,0.857093,0.859990,0.104239,0.860704,0.722561,0.720555,0.860704,0.863685
5,6,Support Vector Machine,0.862712,0.895484,0.826555,0.900139,0.859640,0.862563,0.099861,0.863347,0.727943,0.725712,0.863347,1.405613
6,7,Support Vector Machine,0.862028,0.889173,0.828279,0.896019,0.857647,0.861484,0.103981,0.862149,0.725835,0.724120,0.862149,1.238218
7,8,Support Vector Machine,0.871944,0.895726,0.839144,0.904133,0.866512,0.871032,0.095867,0.871638,0.745177,0.743714,0.871638,1.686471
8,9,Support Vector Machine,0.853308,0.879677,0.818989,0.887710,0.848249,0.852657,0.112290,0.853349,0.708332,0.706640,0.853349,1.358183
9,10,Support Vector Machine,0.865276,0.889349,0.827091,0.901738,0.857091,0.863609,0.098262,0.864415,0.731702,0.729982,0.864415,0.782438


time: 26.1 s (started: 2026-05-09 16:23:14 +00:00)


In [ ]:
from lightgbm import LGBMClassifier

name = "LightGBM" # change as per model

# Model
model = LGBMClassifier(n_jobs=2, verbose=-1) # change as per model

# Parameters, change as per model
param_dist = {
        # 'n_estimators': [21, 51, 101],
        # 'learning_rate': [0.01, 0.05, 0.08],
        # 'num_leaves': [30, 50, 70],
        # 'max_depth': [-1, 5, 10],
        # 'subsample': [0.2, 0.5, 0.8],
        # 'colsample_bytree': [0.2, 0.5, 0.8]
    }

best_model = hyper_parameter_tune(model, param_dist, X_train_sm, y_train_sm)

# run
df4 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df4.to_csv(f"{name} result.csv")
df4

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=5. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters: {}
Best CV accuracy: 0.8990
Test accuracy: 0.8786

Train Model: LightGBM



,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,LightGBM,0.898102,0.921260,0.872542,0.924112,0.896240,0.897957,0.075888,0.898327,0.797418,0.796284,0.898327,0.917821
1,2,LightGBM,0.906651,0.930166,0.880109,0.933402,0.904445,0.906364,0.066598,0.906755,0.814527,0.813338,0.906755,0.904010
2,3,LightGBM,0.913660,0.937225,0.884309,0.942269,0.909998,0.912829,0.057731,0.913289,0.828459,0.827169,0.913289,0.916114
3,4,LightGBM,0.907847,0.934703,0.875902,0.939456,0.904348,0.907123,0.060544,0.907679,0.817231,0.815629,0.907679,1.346737
4,5,LightGBM,0.910241,0.937098,0.882531,0.938846,0.908996,0.910253,0.061154,0.910689,0.822043,0.820606,0.910689,2.755135
5,6,LightGBM,0.905454,0.935925,0.873950,0.938065,0.903876,0.905440,0.061935,0.906007,0.812915,0.811071,0.906007,0.922081
6,7,LightGBM,0.905625,0.929989,0.878024,0.933425,0.903260,0.905301,0.066575,0.905724,0.812564,0.811285,0.905724,0.875983
7,8,LightGBM,0.913489,0.933624,0.888505,0.938008,0.910506,0.912921,0.061992,0.913257,0.827836,0.826886,0.913257,0.877901
8,9,LightGBM,0.897076,0.924453,0.865096,0.929134,0.893790,0.896543,0.070866,0.897115,0.795814,0.794168,0.897115,0.877572
9,10,LightGBM,0.906822,0.930380,0.874694,0.937500,0.901678,0.905553,0.062500,0.906097,0.814691,0.813292,0.906097,0.894401


time: 28.9 s (started: 2026-05-09 16:23:40 +00:00)


In [ ]:
from sklearn.linear_model import LogisticRegression

name ="Logistic Regression" # change as per model

# Model
model = LogisticRegression(random_state=42, max_iter=100, solver="liblinear", C = 1.0) # change as per model

# Parameters, change as per model
param_dist = {
        'C': [0.1, 0.6, 0.8, 1],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
}

best_model = hyper_parameter_tune(model, param_dist, X_train_sm, y_train_sm)

# run
df6 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df6.to_csv(f"{name} result.csv")
df6

Best parameters: {'solver': 'liblinear', 'penalty': 'l2', 'C': 0.8}
Best CV accuracy: 0.8559
Test accuracy: 0.8449

Train Model: Logistic Regression



,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,Logistic Regression,0.857412,0.883611,0.826102,0.889272,0.853889,0.857105,0.110728,0.857687,0.716509,0.714965,0.857687,1.092275
1,2,Logistic Regression,0.863224,0.884726,0.836512,0.890148,0.859944,0.862913,0.109852,0.863330,0.727593,0.726504,0.863330,1.204150
2,3,Logistic Regression,0.870234,0.892910,0.837548,0.902093,0.864343,0.869221,0.097907,0.869820,0.741615,0.740222,0.869820,1.220864
3,4,Logistic Regression,0.856557,0.880795,0.822963,0.889796,0.850897,0.855727,0.110204,0.856380,0.714533,0.713006,0.856380,1.273707
4,5,Logistic Regression,0.862199,0.889809,0.831706,0.893676,0.859777,0.862135,0.106324,0.862691,0.726215,0.724618,0.862691,1.418101
5,6,Logistic Regression,0.863908,0.893748,0.831261,0.897704,0.861372,0.863843,0.102296,0.864482,0.729915,0.728070,0.864482,0.984779
6,7,Logistic Regression,0.863054,0.886314,0.834072,0.892244,0.859400,0.862668,0.107756,0.863158,0.727435,0.726162,0.863158,0.944308
7,8,Logistic Regression,0.870918,0.891734,0.841560,0.899729,0.865921,0.870159,0.100271,0.870645,0.742834,0.741678,0.870645,1.481013
8,9,Logistic Regression,0.855189,0.877951,0.825478,0.884971,0.850907,0.854707,0.115029,0.855225,0.711673,0.710398,0.855225,1.483340
9,10,Logistic Regression,0.866131,0.886940,0.831992,0.898730,0.858588,0.864717,0.101270,0.865361,0.733093,0.731744,0.865361,1.628689


time: 34.9 s (started: 2026-05-09 16:24:09 +00:00)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
name ="Decision Tree"
# Model
model =DecisionTreeClassifier(random_state=42)
# Parameters, change as per model
param_dist = {
        'criterion': ['gini', 'entropy'],
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
}
best_model = hyper_parameter_tune(model, param_dist, X_train_sm, y_train_sm)

# run
df7 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df7.to_csv(f"{name} result.csv")
df7

Best parameters: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'gini'}
Best CV accuracy: 0.8957
Test accuracy: 0.8307

Train Model: Decision Tree



,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,Decision Tree,0.898273,0.890807,0.909831,0.886513,0.900218,0.898096,0.113487,0.898172,0.796678,0.796493,0.898172,0.498266
1,2,Decision Tree,0.902035,0.892655,0.914850,0.889118,0.903616,0.901892,0.110882,0.901984,0.804297,0.804047,0.901984,0.517319
2,3,Decision Tree,0.902889,0.887404,0.919986,0.886226,0.903401,0.902948,0.113774,0.903106,0.806367,0.805837,0.903106,0.757869
3,4,Decision Tree,0.899299,0.889001,0.911310,0.887415,0.900017,0.899283,0.112585,0.899362,0.798868,0.798619,0.899362,0.765009
4,5,Decision Tree,0.908189,0.897973,0.924268,0.891591,0.910931,0.907783,0.108409,0.907930,0.816607,0.816245,0.907930,0.798371
5,6,Decision Tree,0.902718,0.893137,0.918655,0.886221,0.905717,0.902293,0.113779,0.902438,0.805622,0.805281,0.902438,0.694816
6,7,Decision Tree,0.902035,0.891838,0.915843,0.888126,0.903681,0.901878,0.111874,0.901985,0.804341,0.804048,0.901985,0.556305
7,8,Decision Tree,0.910583,0.894352,0.929237,0.892276,0.911461,0.910569,0.107724,0.910757,0.821829,0.821215,0.910757,0.585907
8,9,Decision Tree,0.902376,0.893227,0.914276,0.890448,0.903629,0.902284,0.109552,0.902362,0.804971,0.804747,0.902362,0.554587
9,10,Decision Tree,0.901180,0.886927,0.914246,0.888703,0.900379,0.901384,0.111297,0.901474,0.802755,0.802391,0.901474,0.557168


time: 15.9 s (started: 2026-05-09 16:24:44 +00:00)


In [ ]:
from sklearn.ensemble import StackingClassifier
name="Stacking"
#Model
model== StackingClassifier(
          cv=3,
        estimators=[
            ('lr', LogisticRegression(random_state=42)),
            ('rf', RandomForestClassifier(random_state=42,
                                          n_estimators=50,
                                          max_depth=10,
                                          n_jobs=-1)),
            ('dt', DecisionTreeClassifier(random_state=42, max_depth=10)),
        ],
        n_jobs=-1,
        passthrough=False,
        verbose=1,
         )
param_dist11 = {
       'passthrough': [False],
        'final_estimator': [LogisticRegression(random_state=42, max_iter=200)]
}
best_model = hyper_parameter_tune(model, param_dist, X_train_sm, y_train_sm)

# run
df8 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df8.to_csv(f"{name} result.csv")
df8

Best parameters: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None, 'criterion': 'gini'}
Best CV accuracy: 0.8957
Test accuracy: 0.8307

Train Model: Stacking



,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,Stacking,0.898273,0.890807,0.909831,0.886513,0.900218,0.898096,0.113487,0.898172,0.796678,0.796493,0.898172,0.487638
1,2,Stacking,0.902035,0.892655,0.914850,0.889118,0.903616,0.901892,0.110882,0.901984,0.804297,0.804047,0.901984,0.581709
2,3,Stacking,0.902889,0.887404,0.919986,0.886226,0.903401,0.902948,0.113774,0.903106,0.806367,0.805837,0.903106,0.572255
3,4,Stacking,0.899299,0.889001,0.911310,0.887415,0.900017,0.899283,0.112585,0.899362,0.798868,0.798619,0.899362,0.556260
4,5,Stacking,0.908189,0.897973,0.924268,0.891591,0.910931,0.907783,0.108409,0.907930,0.816607,0.816245,0.907930,0.523738
5,6,Stacking,0.902718,0.893137,0.918655,0.886221,0.905717,0.902293,0.113779,0.902438,0.805622,0.805281,0.902438,0.493498
6,7,Stacking,0.902035,0.891838,0.915843,0.888126,0.903681,0.901878,0.111874,0.901985,0.804341,0.804048,0.901985,0.543292
7,8,Stacking,0.910583,0.894352,0.929237,0.892276,0.911461,0.910569,0.107724,0.910757,0.821829,0.821215,0.910757,0.539998
8,9,Stacking,0.902376,0.893227,0.914276,0.890448,0.903629,0.902284,0.109552,0.902362,0.804971,0.804747,0.902362,0.573834
9,10,Stacking,0.901180,0.886927,0.914246,0.888703,0.900379,0.901384,0.111297,0.901474,0.802755,0.802391,0.901474,0.545869


time: 16.4 s (started: 2026-05-09 16:25:00 +00:00)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
name ="KNN"

model1 = KNeighborsClassifier(algorithm='ball_tree',  n_jobs= -1 )

param_dist1 = {
    "n_neighbors": [1, 3, 5, 7], # Wider range to find optimal balance
    "weights": ["distance"],
    "metric": ["manhattan"],     # Manhattan often performs better in high dimensions
    "leaf_size": [10, 20]
}

best_model = hyper_parameter_tune(model1, param_dist1, X_train_sm, y_train_sm)

# run
df9 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df9.to_csv(f"{name} result.csv")
df9

Best parameters: {'weights': 'distance', 'n_neighbors': 3, 'metric': 'manhattan', 'leaf_size': 10}
Best CV accuracy: 0.9268
Test accuracy: 0.8373

Train Model: KNN



,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,KNN,0.927851,0.901270,0.962373,0.892722,0.930820,0.926893,0.107278,0.927547,0.857615,0.855605,0.927547,0.360687
1,2,KNN,0.930757,0.904701,0.963556,0.897700,0.933201,0.930045,0.102300,0.930628,0.863321,0.861477,0.930628,0.625448
2,3,KNN,0.930074,0.900712,0.964669,0.896354,0.931594,0.929884,0.103646,0.930512,0.862368,0.860250,0.930512,0.394908
3,4,KNN,0.924944,0.893814,0.963561,0.886735,0.927378,0.924350,0.113265,0.925148,0.852516,0.849946,0.925148,0.455765
4,5,KNN,0.930415,0.907243,0.961292,0.898541,0.933486,0.929387,0.101459,0.929917,0.862246,0.860662,0.929917,0.485587
5,6,KNN,0.932125,0.906112,0.966723,0.896312,0.935437,0.930852,0.103688,0.931517,0.866063,0.864050,0.931517,0.483096
6,7,KNN,0.931441,0.900696,0.970358,0.892244,0.934230,0.930482,0.107756,0.931301,0.865455,0.862843,0.931301,0.646388
7,8,KNN,0.934006,0.905131,0.968243,0.900407,0.935624,0.933709,0.099593,0.934325,0.870162,0.868086,0.934325,0.359624
8,9,KNN,0.927509,0.898219,0.964481,0.890448,0.930171,0.926726,0.109552,0.927465,0.857347,0.855005,0.927465,0.484841
9,10,KNN,0.929903,0.898664,0.965348,0.896056,0.930813,0.930057,0.103944,0.930702,0.862228,0.859965,0.930702,0.358711


time: 2min 17s (started: 2026-05-09 16:25:16 +00:00)


In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

name="BaggingClassifier"

model2 =   BaggingClassifier(n_estimators=10)
param_dist2 = {
     "n_estimators": [50, 100]
}
best_model = hyper_parameter_tune(model2, param_dist2, X_train_sm, y_train_sm)

# run
df10 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df10.to_csv(f"{name} result.csv")
df10

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=5. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters: {'n_estimators': 100}
Best CV accuracy: 0.9311
Test accuracy: 0.8731

Train Model: BaggingClassifier



,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,BaggingClassifier,0.928706,0.929468,0.929153,0.928251,0.929310,0.928702,0.071749,0.928702,0.857401,0.857401,0.928702,38.287507
1,2,BaggingClassifier,0.937938,0.940431,0.935627,0.940268,0.938023,0.937944,0.059732,0.937947,0.875888,0.875877,0.937947,37.958758
2,3,BaggingClassifier,0.941699,0.942014,0.939730,0.943619,0.940870,0.941672,0.056381,0.941674,0.883379,0.883376,0.941674,52.514671
3,4,BaggingClassifier,0.928706,0.930249,0.926091,0.931293,0.928165,0.928688,0.068707,0.928692,0.857413,0.857404,0.928692,38.227942
4,5,BaggingClassifier,0.937254,0.940162,0.936048,0.938499,0.938101,0.937273,0.061501,0.937274,0.874494,0.874486,0.937274,38.092425
5,6,BaggingClassifier,0.934861,0.933489,0.938824,0.930759,0.936149,0.934782,0.069241,0.934791,0.869684,0.869669,0.934791,40.769087
6,7,BaggingClassifier,0.937254,0.934959,0.940375,0.934111,0.937659,0.937238,0.065889,0.937243,0.874519,0.874504,0.937243,56.406581
7,8,BaggingClassifier,0.946487,0.947989,0.943735,0.949187,0.945857,0.946457,0.050813,0.946461,0.892968,0.892959,0.946461,37.305851
8,9,BaggingClassifier,0.931954,0.929980,0.934426,0.929476,0.932198,0.931948,0.070524,0.931951,0.863917,0.863907,0.931951,38.050783
9,10,BaggingClassifier,0.934861,0.933170,0.933497,0.936163,0.933333,0.934829,0.063837,0.934830,0.869653,0.869653,0.934830,37.070760


time: 11min 46s (started: 2026-05-09 16:27:33 +00:00)


In [ ]:
from xgboost import XGBClassifier

name = "XGBoost"

# Model
model = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

# Parameters
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

best_model = hyper_parameter_tune(model, param_dist, X_train_sm, y_train_sm)

# run
df11 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df11.to_csv(f"{name} result.csv")
df11

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:40:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best parameters: {'subsample': 1.0, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
Best CV accuracy: 0.8889
Test accuracy: 0.8792

Train Model: XGBoost



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:40:19] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:40:22] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:40:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:40:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,XGBoost,0.888528,0.921188,0.851864,0.925836,0.885171,0.888081,0.074164,0.888850,0.779433,0.777186,0.888850,3.318557
1,2,XGBoost,0.896734,0.929624,0.859332,0.934432,0.893097,0.896096,0.065568,0.896882,0.795822,0.793527,0.896882,2.388036
2,3,XGBoost,0.903744,0.933582,0.866644,0.939905,0.898868,0.902531,0.060095,0.903275,0.809316,0.807282,0.903275,2.231829
3,4,XGBoost,0.897589,0.929368,0.859402,0.935374,0.893017,0.896584,0.064626,0.897388,0.797332,0.795092,0.897388,4.495567
4,5,XGBoost,0.897589,0.934114,0.858970,0.937457,0.894968,0.897356,0.062543,0.898213,0.798057,0.795393,0.898213,2.394540
5,6,XGBoost,0.895367,0.935175,0.853445,0.938761,0.892443,0.895087,0.061239,0.896103,0.794129,0.790992,0.896103,2.219561
6,7,XGBoost,0.893657,0.925653,0.856899,0.930679,0.889950,0.893028,0.069321,0.893789,0.789564,0.787368,0.893789,2.410761
7,8,XGBoost,0.904428,0.933284,0.869175,0.939024,0.900089,0.903425,0.060976,0.904100,0.810598,0.808717,0.904100,4.881745
8,9,XGBoost,0.889041,0.922820,0.849385,0.928792,0.884581,0.888201,0.071208,0.889088,0.780583,0.778103,0.889088,3.379275
9,10,XGBoost,0.898102,0.929358,0.856493,0.937834,0.891439,0.896241,0.062166,0.897164,0.798090,0.795735,0.897164,2.192993


time: 1min 29s (started: 2026-05-09 16:39:20 +00:00)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

name = "Random Forest"

# Model
model = RandomForestClassifier(
    random_state=42
)

# Parameters
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

best_model = hyper_parameter_tune(model, param_dist, X_train_sm, y_train_sm)

# run
df12 = evaluate_models(name, best_model, X_train_sm, y_train_sm)

df12.to_csv(f"{name} result.csv")
df12

Best parameters: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 10}
Best CV accuracy: 0.8290
Test accuracy: 0.8605

Train Model: Random Forest



,Fold,Classifier,Accuracy,Precision,Recall,Specificity,F1,GM,FPR,AUC,MCC,Kappa,Balanced Accuracy,Training Time (s)
0,1,Random Forest,0.817747,0.872038,0.748475,0.888237,0.805545,0.815367,0.111763,0.818356,0.642501,0.635919,0.818356,8.554156
1,2,Random Forest,0.833647,0.884751,0.768733,0.899073,0.822672,0.831353,0.100927,0.833903,0.673315,0.667461,0.833903,8.960612
2,3,Random Forest,0.842879,0.888933,0.779009,0.905132,0.830349,0.839706,0.094868,0.842071,0.690395,0.685214,0.842071,9.361779
3,4,Random Forest,0.828518,0.881505,0.756961,0.899320,0.814500,0.825076,0.100680,0.828140,0.663364,0.656770,0.828140,9.009765
4,5,Random Forest,0.825611,0.884813,0.754965,0.898541,0.814748,0.823630,0.101459,0.826753,0.659296,0.651959,0.826753,8.115104
5,6,Random Forest,0.826124,0.891287,0.749580,0.905358,0.814314,0.823795,0.094642,0.827469,0.661783,0.653114,0.827469,9.554519
6,7,Random Forest,0.823218,0.874951,0.755707,0.891215,0.810969,0.820669,0.108785,0.823461,0.652733,0.646604,0.823461,9.258565
7,8,Random Forest,0.842024,0.883405,0.784605,0.898374,0.831079,0.839564,0.101626,0.841489,0.687936,0.683691,0.841489,8.506209
8,9,Random Forest,0.820995,0.871886,0.753074,0.889079,0.808136,0.818256,0.110921,0.821076,0.648106,0.642048,0.821076,8.873946
9,10,Random Forest,0.836040,0.882661,0.766188,0.902741,0.820311,0.831667,0.097259,0.834464,0.676612,0.670926,0.834464,9.826267


time: 3min 16s (started: 2026-05-09 16:40:50 +00:00)


In [ ]:
# Combine all results into ONE Excel file (multiple sheets)

with pd.ExcelWriter("Final_Result(Smote).xlsx") as writer:
    df1.to_cvs(writer, sheet_name="Gradient Boosting (GBM)", index=False)
    df2.to_cvs(writer, sheet_name="Multi Layer Perceptron", index=False)
    df3.to_cvs(writer, sheet_name="Support Vector Machine", index=False)
    df4.to_cvs(writer, sheet_name="LightGBM", index=False)
    df5.to_cvs(writer, sheet_name="CatBoost", index=False)
    df6.to_cvs(writer, sheet_name="Logistic Regression", index=False)
    df7.to_cvs(writer, sheet_name="Decision Tree", index=False)
    df8.to_cvs(writer, sheet_name="Stacking", index=False)
    df9.to_cvs(writer, sheet_name="KNN", index=False)
    df10.to_cvs(writer, sheet_name="Bagging", index=False)
    df11.to_cvs(writer, sheet_name="XGBoost", index=False)
    df12.to_cvs(writer, sheet_name="Random Forest", index=False)


print("Excel file created successfully!")

Excel file created successfully!
time: 157 ms (started: 2026-05-09 16:44:06 +00:00)


In [ ]:
from google.colab import files
files.download("Final_Result(Smote).xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

time: 7.9 ms (started: 2026-05-09 16:44:06 +00:00)


In [ ]:
import pandas as pd
from scipy.stats import friedmanchisquare

# Load Excel file
file_path = "Final_Result(Smote).xlsx"
xls = pd.ExcelFile(file_path)

model_scores = []
model_names = []

for sheet in xls.sheet_names:

    df = pd.read_excel(file_path, sheet_name=sheet)

    scores = df["Accuracy"].values

    model_scores.append(scores)
    model_names.append(sheet)

print("Models:")
for m in model_names:
    print(m)

stat, p = friedmanchisquare(*model_scores)

print("\nFriedman Statistic:", stat)
print("P-value:", p)

if p < 0.05:
    print("Significant difference exists among models")
else:
    print("No significant difference among models")

Models:
Gradient Boosting (GBM)
Multi Layer Perceptron
Support Vector Machine
LightGBM
CatBoost
Logistic Regression
Decision Tree
Stacking
KNN
Bagging
XGBoost
Random Forest

Friedman Statistic: 104.33778089887645
P-value: 2.4614347936378533e-17
Significant difference exists among models
time: 922 ms (started: 2026-05-09 16:44:06 +00:00)
